# POEM dual-T4 Kaggle training

Attach two Kaggle datasets before running:

1. `POEM-BASE`: a mirror of this GitHub repository
2. `Beautiful-Motifs-CC-BY-NC-SA`: the MIDI dataset

Set the Kaggle accelerator to **GPU T4 x2**. The workflow pretokenizes short motifs, skips already-finished D/C/E, trains variants F/B/A, generates five MIDI samples per completed variant, and uploads each completed model folder to a Hugging Face model repo named `POEM-BASE` under your namespace.

In [ ]:
!nvidia-smi
!python --version
!pip install -q pretty_midi mido huggingface_hub einops
!pip install -q "flash-linear-attention[cuda]"
from fla.layers import GatedDeltaNet
print('FLA GatedDeltaNet import OK')

In [ ]:
from pathlib import Path
import os, shutil, json

def first_existing(candidates):
    for path in candidates:
        p = Path(path)
        if p.exists():
            return p
    return None

def find_repo_dir(input_root):
    explicit = first_existing([
        input_root / 'poem-base',
        input_root / 'POEM-BASE',
        input_root / 'poem',
        input_root / 'POEM',
        input_root / 'datasets' / 'chickaboomcmurtrie' / 'poem-base',
    ])
    if explicit is not None:
        return explicit
    matches = []
    for p in input_root.rglob('*'):
        if p.is_dir() and (p / 'train.py').exists() and (p / 'models').exists() and (p / 'tokenizer').exists():
            matches.append(p)
    if not matches:
        print('Available input directories:')
        for p in sorted([x for x in input_root.rglob('*') if x.is_dir()])[:200]:
            print(' ', p)
        raise FileNotFoundError('Could not find POEM-BASE repo dataset. Expected train.py plus models/ and tokenizer/.')
    return sorted(matches, key=lambda p: len(p.parts))[0]

def find_motif_data_dir(input_root):
    explicit = first_existing([
        input_root / 'beautiful-motifs-cc-by-nc-sa',
        input_root / 'Beautiful-Motifs-CC-BY-NC-SA',
        input_root / 'datasets' / 'chickaboomcmurtrie' / 'beautiful-motifs-cc-by-nc-sa',
    ])
    if explicit is not None and (explicit / 'MIDIs').exists():
        return explicit
    matches = []
    for p in input_root.rglob('*'):
        if p.is_dir() and (p / 'MIDIs').exists():
            matches.append(p)
    if not matches:
        raise FileNotFoundError('Could not find Beautiful-Motifs dataset folder containing MIDIs/.')
    return sorted(matches, key=lambda p: len(p.parts))[0]

INPUT = Path('/kaggle/input')
repo_src = find_repo_dir(INPUT)
repo_dst = Path('/kaggle/working/POEM-BASE')
if repo_dst.exists():
    shutil.rmtree(repo_dst)
shutil.copytree(repo_src, repo_dst, ignore=shutil.ignore_patterns('.git', '__pycache__', '*.pyc'))
os.chdir(repo_dst)
print('Repo:', repo_dst)

data_dir = find_motif_data_dir(INPUT)
print('Data:', data_dir)

In [ ]:
from getpass import getpass
import os

HF_TOKEN = getpass('Paste a Hugging Face token with write access to your private POEM-BASE repo: ')
os.environ['HF_TOKEN'] = HF_TOKEN
HF_NAMESPACE = input('Hugging Face username or org: ').strip()
HF_REPO_ID = f'{HF_NAMESPACE}/POEM-BASE'
print('HF repo:', HF_REPO_ID)

In [ ]:
# Tune these if the 12-hour session is tight. The script stops before starting a new variant once max_hours is reached.
EPOCHS = 40
BATCH_SIZE_C = 256
BATCH_SIZE_E = 256
BATCH_SIZE_B = 64
BATCH_SIZE_A = 32
BATCH_SIZE_F = 128
MAX_HOURS = 11.75
PRETOKENIZE_WORKERS = 8
VARIANTS = 'F B A'

cmd = f"""
python -u scripts/kaggle_train_all.py \
  --repo_dir /kaggle/working/POEM-BASE \
  --data_dir '{data_dir}' \
  --hf_repo_id '{HF_REPO_ID}' \
  --private \
  --epochs {EPOCHS} \
  --batch_size_c {BATCH_SIZE_C} \
  --batch_size_e {BATCH_SIZE_E} \
  --batch_size_b {BATCH_SIZE_B} \
  --batch_size_a {BATCH_SIZE_A} \
  --batch_size_f {BATCH_SIZE_F} \
  --variants {VARIANTS} \
  --max_hours {MAX_HOURS} \
  --val_interval 2000 \
  --checkpoint_interval_steps 5000 \
  --checkpoint_interval_minutes 20 \
  --num_workers 2 \
  --pretokenize_workers {PRETOKENIZE_WORKERS}
"""
print(cmd)
!{cmd}

In [ ]:
from pathlib import Path
summary = Path('/kaggle/working/comparison/summary.json')
if summary.exists():
    print(summary.read_text()[:4000])
else:
    print('No comparison summary found yet.')
!find /kaggle/working -maxdepth 3 -type f | sed 's#/kaggle/working/##' | sort | head -200